由於訓練是在 Colab GPU 環境執行，請確認 Google Drive 中的資料集目錄 `/content/drive/MyDrive/SO101/datasets/so101_pick_cube` 已包含 `data/`、`videos/`、`meta/`。

> **斷點續訓與自動存檔說明**：本 Notebook 專用於訓練 **`so101_pick_cube`**，並已內建完整斷點續訓（包含模型權重、前後處理器、Optimizer、LR Scheduler、Loss 紀錄與隨機種子），預設每 **500 steps** 透過本地暫存與原子同步更新 Google Drive 的 `outputs/smolvla/so101_pick_cube/checkpoints/latest`（目前 `MAX_SESSION_MINUTES = None` 不限制單次執行時間），並在完成全部訓練後自動驗證雲端存檔、刷新 Drive 快取並中斷 Colab 連線；若中途手動停止或意外中斷，下次重啟環境後直接由上而下執行所有 Cell 即可從最新進度接續訓練。

# 環境變數與模型權重下載（與 MT05 一致）
保留原本範例針對 AMD ROCm 環境的設定，先連網快取 `lerobot/smolvla_base` 與其 SmolVLM2 骨幹後切換為離線模式，並釋放暫存模型以節省記憶體：

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q "lerobot[smolvla,dataset]==0.7.0" av
import lerobot.utils.import_utils as _iu
_iu._require_package_cache.clear()

In [ ]:
import gc
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import glob
import random
import shutil
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from huggingface_hub import snapshot_download
from lerobot.datasets.lerobot_dataset import LeRobotDataset, LeRobotDatasetMetadata
from lerobot.configs.policies import PreTrainedConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.policies.factory import make_policy, make_pre_post_processors

if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive 尚未掛載，請先執行上方 drive.mount('/content/drive') Cell！")

PROJECT_DIR = "/content/drive/MyDrive/SO101"
DATASET_NAME = "so101_pick_cube"
CHECKPOINT_ROOT = os.path.join(PROJECT_DIR, "outputs", "smolvla", DATASET_NAME, "checkpoints")
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "| torch:", torch.__version__)

BASE = "lerobot/smolvla_base"
snapshot_download(BASE)
_tmp_policy = SmolVLAPolicy.from_pretrained(BASE)
del _tmp_policy
gc.collect()
os.environ["HF_HUB_OFFLINE"] = "1"
print("SmolVLA base ready")

# 載入實體 SO-101 雙相機資料集、適配 SmolVLA 並偵測斷點
此處已修正以下關鍵問題：
1. **相機特徵適配**：清空 `smolvla_base` 預設的 `camera1/2/3` 槽位（設 `cfg.input_features = None`），讓 `make_policy` 自動依資料集建立 `observation.state`、`observation.images.wrist` 與 `observation.images.side_45`，避免訓練階段 `prepare_images` 找不到影像而崩潰，也讓本地部署時不需 `--rename_map`。
2. **正規化統計量覆寫**：透過 `preprocessor_overrides` / `postprocessor_overrides` 明確注入 `ds.meta.stats`，確保 SO-101 的 6 軸關節角度有正確執行 `MEAN_STD` 正規化與反正規化。
3. **自動偵測斷點**：若 `outputs/smolvla/so101_pick_cube/checkpoints/` 已有中斷前的完整 checkpoint（`latest` 或 `smolvla_pick_step_*`），會自動載入進度最高的權重與訓練狀態。
4. **可訓練權重 FP32 主複本與舊版 BF16 斷點檢查**：在建立 Optimizer 前將所有 `requires_grad=True` 的可訓練參數（包含原為 `bfloat16` 的 `lm_expert` 權重與 LayerNorm）轉為 `float32`，避免 AdamW 小步長更新被 BF16 精度捨入抹除（Update Erasure）；並自動跳過受 BF16 捨入影響的舊版 checkpoint，確保重新微調生效。

In [ ]:
if not os.path.ismount("/content/drive"):
    raise RuntimeError("Google Drive 尚未掛載，請先執行上方 drive.mount('/content/drive') Cell！")

PROJECT_DIR = "/content/drive/MyDrive/SO101"
DATASET_NAME = "so101_pick_cube"
REPO_ID = f"local/{DATASET_NAME}"
DS_ROOT = f"/content/drive/MyDrive/SO101/datasets/so101_pick_cube"

DRIVE_CHECKPOINT_ROOT = os.path.join(PROJECT_DIR, "outputs", "smolvla", DATASET_NAME, "checkpoints")
LOCAL_CHECKPOINT_ROOT = os.path.join("/content/outputs/smolvla", DATASET_NAME, "checkpoints")
CHECKPOINT_ROOT = DRIVE_CHECKPOINT_ROOT
LATEST_DIR = os.path.join(DRIVE_CHECKPOINT_ROOT, "latest")
os.makedirs(DRIVE_CHECKPOINT_ROOT, exist_ok=True)
os.makedirs(LOCAL_CHECKPOINT_ROOT, exist_ok=True)
AUTO_RESUME = True  # 若要從頭重訓，請改為 False 或清空 outputs/smolvla/checkpoints

REQUIRED_CKPT_FILES = (
    "training_state.pt",
    "config.json",
    "model.safetensors",
    "policy_preprocessor.json",
    "policy_postprocessor.json",
)

if not os.path.isfile(os.path.join(DS_ROOT, "meta", "info.json")):
    raise FileNotFoundError(f"找不到資料集或 meta/info.json：{DS_ROOT}")

def verify_checkpoint_dir(ckpt_dir):
    """確認 checkpoint 目錄內必要檔案與 processor 狀態檔皆存在且大小不為 0。"""
    import json
    if not os.path.isdir(ckpt_dir):
        return False
    for fname in REQUIRED_CKPT_FILES:
        fpath = os.path.join(ckpt_dir, fname)
        if not os.path.isfile(fpath) or os.path.getsize(fpath) == 0:
            return False
    for cfg_name in ("policy_preprocessor.json", "policy_postprocessor.json"):
        try:
            with open(os.path.join(ckpt_dir, cfg_name), "r", encoding="utf-8") as f:
                proc_cfg = json.load(f)
            for step_cfg in proc_cfg.get("steps", []):
                state_file = step_cfg.get("state_file")
                if state_file:
                    s_path = os.path.join(ckpt_dir, state_file)
                    if not os.path.isfile(s_path) or os.path.getsize(s_path) == 0:
                        return False
        except Exception:
            return False
    return True

def recover_atomic_checkpoints(checkpoint_root):
    """若前次原子寫入中途被強制終止，清理暫存目錄並還原備份。"""
    if not os.path.isdir(checkpoint_root):
        return
    for name in os.listdir(checkpoint_root):
        full_path = os.path.join(checkpoint_root, name)
        if name.startswith("_") and (name.endswith("_tmp") or "_tmp_" in name):
            shutil.rmtree(full_path, ignore_errors=True)
    for name in sorted(os.listdir(checkpoint_root), reverse=True):
        if name.startswith("_") and (name.endswith("_old") or "_old_" in name):
            orig_name = name[1:].rsplit("_old", 1)[0]
            old_dir = os.path.join(checkpoint_root, name)
            orig_dir = os.path.join(checkpoint_root, orig_name)
            if not os.path.exists(orig_dir):
                os.rename(old_dir, orig_dir)
            else:
                shutil.rmtree(old_dir, ignore_errors=True)

def is_legacy_bf16_checkpoint(ckpt_dir):
    """確認 SmolVLA checkpoint 的可訓練 lm_expert 是否仍為舊版 BF16（會導致梯度更新捨入歸零）。"""
    import json
    import struct
    cfg_path = os.path.join(ckpt_dir, "config.json")
    weights_path = os.path.join(ckpt_dir, "model.safetensors")
    if not os.path.isfile(cfg_path) or not os.path.isfile(weights_path):
        return False
    try:
        with open(cfg_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)
        if cfg.get("type") != "smolvla":
            return False
        with open(weights_path, "rb") as f:
            header_len = struct.unpack("<Q", f.read(8))[0]
            header = json.loads(f.read(header_len).decode("utf-8"))
        norm_meta = header.get("model.vlm_with_expert.lm_expert.norm.weight")
        return bool(norm_meta and norm_meta.get("dtype") == "BF16")
    except Exception:
        return False

def find_latest_checkpoint(*checkpoint_roots):
    """掃描所有完整 checkpoint，回傳 step 數最高的 (目錄路徑, training_state dict)。"""
    best_dir, best_ckpt, best_step = None, None, -1
    for root in checkpoint_roots:
        if not os.path.isdir(root):
            continue
        for name in os.listdir(root):
            if name.startswith("_"):
                continue
            cand_dir = os.path.join(root, name)
            if not verify_checkpoint_dir(cand_dir):
                continue
            if is_legacy_bf16_checkpoint(cand_dir):
                print(f"Warning: skipping legacy BF16-underflow SmolVLA checkpoint {cand_dir} (starting fresh FP32 fine-tuning).")
                continue
            try:
                ckpt = torch.load(os.path.join(cand_dir, "training_state.pt"), map_location="cpu", weights_only=False)
                ckpt_step = int(ckpt.get("step", -1))
                if ckpt_step > best_step or (ckpt_step == best_step and name == "latest"):
                    best_step = ckpt_step
                    best_dir = cand_dir
                    best_ckpt = ckpt
            except Exception as e:
                print(f"Warning: skipping corrupted checkpoint {cand_dir}: {e}")
    return best_dir, best_ckpt

recover_atomic_checkpoints(DRIVE_CHECKPOINT_ROOT)
recover_atomic_checkpoints(LOCAL_CHECKPOINT_ROOT)

# 使用 LeRobotDatasetMetadata 輕量讀取 FPS，再載入帶有 Action Chunk 的資料集
ds_meta = LeRobotDatasetMetadata(REPO_ID, root=DS_ROOT)
FPS = ds_meta.fps
CHUNK = 50
delta_timestamps = {"action": [i / FPS for i in range(CHUNK)]}
ds = LeRobotDataset(REPO_ID, root=DS_ROOT, delta_timestamps=delta_timestamps, video_backend="pyav")
print(f"Loaded dataset: {ds.num_episodes} episodes, {ds.num_frames} frames @ {FPS} FPS")
print("Dataset cameras:", ds.meta.camera_keys)

resume_dir, resume_ckpt = (
    find_latest_checkpoint(DRIVE_CHECKPOINT_ROOT, LOCAL_CHECKPOINT_ROOT)
    if AUTO_RESUME else (None, None)
)

if resume_dir is not None:
    print(f"--> Resuming from checkpoint: {resume_dir} (step {resume_ckpt['step']})")
    cfg = PreTrainedConfig.from_pretrained(resume_dir)
    cfg.pretrained_path = resume_dir
    cfg.device = device
    policy = make_policy(cfg, ds_meta=ds.meta)
    preprocess, postprocess = make_pre_post_processors(
        policy.config,
        pretrained_path=resume_dir,
        preprocessor_overrides={"device_processor": {"device": device}},
    )
else:
    print(f"--> Starting fresh fine-tune from base model: {BASE}")
    cfg = PreTrainedConfig.from_pretrained(BASE)
    cfg.pretrained_path = BASE
    cfg.device = device
    # 清空預訓練預設相機槽位 (camera1/2/3)，交由 make_policy 依 ds.meta 自動推導 state + top + front
    cfg.input_features = None

    policy = make_policy(cfg, ds_meta=ds.meta)
    preprocess, postprocess = make_pre_post_processors(
        policy.config,
        pretrained_path=BASE,
        preprocessor_overrides={
            "device_processor": {"device": device},
            "normalizer_processor": {
                "features": {**policy.config.input_features, **policy.config.output_features},
                "norm_map": policy.config.normalization_mapping,
                "stats": ds.meta.stats,
            },
        },
        postprocessor_overrides={
            "unnormalizer_processor": {
                "features": policy.config.output_features,
                "norm_map": policy.config.normalization_mapping,
                "stats": ds.meta.stats,
            },
        },
    )

promoted_fp32 = 0
for p in policy.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
        promoted_fp32 += 1
print("Trainable parameter dtypes:", {str(p.dtype) for p in policy.parameters() if p.requires_grad}, f"(promoted {promoted_fp32} tensors to FP32)")

print("Policy input features:", list(policy.config.input_features.keys()))
print("SmolVLA ready:", round(sum(p.numel() for p in policy.parameters()) / 1e6, 1), "M params")

# 執行完整微調訓練與雲端斷點儲存（支援自動同步至 Drive 並結束工作階段）
- 使用 `SmolVLAConfig` 預設的 AdamW（`betas=(0.9, 0.95)`、`weight_decay=1e-10`、`lr=1e-4`）、Cosine Decay with Warmup LR Scheduler（`warmup=1000`、`decay=30000`、`decay_lr=2.5e-6`，若 `STEPS < 30000` 會自動等比例縮放）與梯度裁剪（`grad_clip_norm=10`）。
- **官方建議訓練參數參考（`docs/source/smolvla.mdx` / `configuration_smolvla.py`）**：
  - **資料集**：建議約 **50 episodes**（同一變因至少 10 集；少於 25 集泛化會明顯下降）。
  - **Batch Size 與步數**：官方預設 **`batch_size = 64`**、**`steps = 20,000 ~ 30,000`**（相當於 `1.28M ~ 1.92M` 總樣本數；若受顯存限制設為 `BATCH_SIZE = 16` 或 `32`，建議對應拉高 `STEPS` 至 `25,000 ~ 30,000` 或搭配梯度累積）。
  - **視覺編碼器（選用）**：預設 `freeze_vision_encoder = True`、`train_expert_only = True`；若目標物細小（如觸控筆）且重訓後仍抓不準，可在 Cell 6 建立模型前設定 `cfg.freeze_vision_encoder = False` 解凍視覺編碼器。
- **優化存檔行為**：先高速寫入本地暫存 `/content/outputs/smolvla/checkpoints/_ckpt_staging`（只序列化一次，避免最後一步重複存檔），再以原子方式（`_tmp` → `os.sync()` → 替換目標）同步至 Google Drive `outputs/smolvla/so101_pick_cube/checkpoints/latest` 與里程碑 `smolvla_pick_step_{step}`，並驗證所有必要檔案完整存在且不為空。
- **自動結束連線**：目前設定 `MAX_SESSION_MINUTES = None`（無單次時間限制）；待全部訓練完成並確認雲端存檔完整後，會執行 `drive.flush_and_unmount()` 確保所有檔案刷新回 Google Drive，再呼叫 `runtime.unassign()` 自動中斷 Colab 執行階段連線。

In [ ]:
# --- 官方 SmolVLA 建議訓練參數（參考 lerobot docs/source/smolvla.mdx & configuration_smolvla.py） ---
# - batch_size: 64（22GB L4 若 OOM 可改 32；小 batch 會降低總見過樣本數，建議同步拉高 STEPS）
# - steps: 20,000 ~ 30,000（配合 scheduler_decay_steps=30,000；bs=16 時 15,000 steps 僅官方總樣本數約 1/5）
# - optimizer: AdamW (lr=1e-4, betas=(0.9, 0.95), eps=1e-8, weight_decay=1e-10, grad_clip_norm=10.0)
# - scheduler: CosineDecayWithWarmup (warmup=1,000, decay=30,000, decay_lr=2.5e-6)
# - dataset: 建議 ~50 episodes；若細小物件定位仍不準，可在 Cell 6 設 cfg.freeze_vision_encoder = False
BATCH_SIZE = 32                         # 官方建議 64；22GB VRAM 建議可試 32（較不易 OOM）
NUM_WORKERS = 0                         # 容器 /dev/shm 預設較小，設為 0 避免 Bus error（若 shm 足夠可改 2~4）
STEPS = 25000                           # 官方建議 20,000 ~ 30,000 steps（若使用 bs=16/32 建議調高至 20,000~30,000）
SAVE_LATEST_FREQ = 500                  # 每 500 steps 原子更新 latest，降低單次中斷損失
MILESTONE_FREQ = 5000                   # 每 5,000 steps 保留一個里程碑 checkpoint
MAX_SESSION_MINUTES = None              # 不限制單次執行時間，直到全部 STEPS 訓練完成（若需限時可改為分鐘數）
LR = 1e-4                               # 官方預設 Peak LR = 1e-4（Cosine 衰減至 2.5e-6）
AUTO_DISCONNECT_ON_COMPLETE = True      # 訓練完成並確認雲端存檔後，自動 flush Drive 並中斷 Colab 連線
AUTO_DISCONNECT_ON_SESSION_LIMIT = True # 達單次時間上限存檔後，自動 flush Drive 並中斷 Colab 連線

opt_cfg = policy.config.get_optimizer_preset()
opt_cfg.lr = LR
grad_clip_norm = opt_cfg.grad_clip_norm
optimizer = opt_cfg.build(policy.get_optim_params())

sched_cfg = policy.config.get_scheduler_preset()
sched_cfg.peak_lr = LR
scheduler = sched_cfg.build(optimizer, num_training_steps=STEPS)

start_step = 0
losses = []

if resume_ckpt is not None:
    start_step = int(resume_ckpt["step"])
    losses = list(resume_ckpt.get("losses", []))
    optimizer.load_state_dict(resume_ckpt["optimizer"])
    for p in policy.parameters():
        if p in optimizer.state:
            for k, v in optimizer.state[p].items():
                if isinstance(v, torch.Tensor) and v.is_floating_point() and k != "step" and v.dtype != p.dtype:
                    optimizer.state[p][k] = v.to(dtype=p.dtype)
    scheduler.load_state_dict(resume_ckpt["scheduler"])
    rng_state = resume_ckpt.get("rng_state")
    if rng_state is not None:
        random.setstate(rng_state["python"])
        np.random.set_state(rng_state["numpy"])
        torch.set_rng_state(rng_state["torch"])
        if torch.cuda.is_available() and rng_state.get("cuda") is not None:
            torch.cuda.set_rng_state_all(rng_state["cuda"])
    print(f"Restored training state at step {start_step}/{STEPS} (loaded {len(losses)} loss records)")

def sync_dir_atomically(src_dir, dst_dir):
    """以暫存目錄 + 原子更名方式將本地完整的 checkpoint 同步到雲端目標目錄。"""
    parent_dir = os.path.dirname(dst_dir)
    base_name = os.path.basename(dst_dir)
    tmp_dir = os.path.join(parent_dir, f"_{base_name}_tmp")
    old_dir = os.path.join(parent_dir, f"_{base_name}_old")
    os.makedirs(parent_dir, exist_ok=True)
    shutil.rmtree(tmp_dir, ignore_errors=True)
    shutil.rmtree(old_dir, ignore_errors=True)
    if os.path.exists(tmp_dir):
        tmp_dir = os.path.join(parent_dir, f"_{base_name}_tmp_{int(time.time() * 1000)}")
    if os.path.exists(old_dir):
        old_dir = os.path.join(parent_dir, f"_{base_name}_old_{int(time.time() * 1000)}")
    shutil.copytree(src_dir, tmp_dir, dirs_exist_ok=True)
    os.sync()
    if os.path.exists(dst_dir):
        os.rename(dst_dir, old_dir)
    os.rename(tmp_dir, dst_dir)
    shutil.rmtree(old_dir, ignore_errors=True)
    os.sync()

def save_checkpoint(target_dirs, current_step):
    """先寫入本地 SSD 暫存並驗證，再原子同步至一或多個雲端目標目錄（避免重複序列化）。"""
    if isinstance(target_dirs, str):
        target_dirs = [target_dirs]
    staging_dir = os.path.join(LOCAL_CHECKPOINT_ROOT, "_ckpt_staging")
    shutil.rmtree(staging_dir, ignore_errors=True)
    os.makedirs(staging_dir, exist_ok=True)

    state_payload = {
        "step": current_step,
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "losses": losses,
        "rng_state": {
            "python": random.getstate(),
            "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }
    try:
        policy.save_pretrained(staging_dir)
        preprocess.save_pretrained(staging_dir)
        postprocess.save_pretrained(staging_dir)
        torch.save(state_payload, os.path.join(staging_dir, "training_state.pt"))
        if not verify_checkpoint_dir(staging_dir):
            raise RuntimeError(f"本地暫存 checkpoint 驗證失敗：{staging_dir}")

        for dst_dir in target_dirs:
            sync_dir_atomically(staging_dir, dst_dir)
            if not verify_checkpoint_dir(dst_dir):
                raise RuntimeError(f"雲端 checkpoint 驗證失敗：{dst_dir}")
    finally:
        shutil.rmtree(staging_dir, ignore_errors=True)

def build_trimmed_subset(
    dataset,
    ds_root,
    move_thresh=3.0,
    keep_lead_frames=10,
    keep_tail_frames=10,
    trim_head=True,
    trim_tail=True,
    drop_longest_n=0,
    exclude_episodes=None,
):
    """排除時間最長／指定 episode，並自動裁切每條 episode 開頭與結尾的靜止等待幀。"""
    files = sorted(glob.glob(f"{ds_root}/data/*/*.parquet"))
    if not files:
        raise FileNotFoundError(f"在 {ds_root}/data/*/*.parquet 找不到任何 parquet 資料檔。")
    df = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)
    sort_cols = [c for c in ("episode_index", "frame_index", "index") if c in df.columns]
    if sort_cols:
        df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    states = np.stack(df["observation.state"].values)
    eps = df["episode_index"].values
    indices = df["index"].values
    timestamps = df["timestamp"].values if "timestamp" in df.columns else None

    unique_eps = np.unique(eps)
    ep_masks = {int(ep): (eps == ep) for ep in unique_eps}

    excluded_set = {int(e) for e in (exclude_episodes or ())}
    remaining_eps = [int(ep) for ep in unique_eps if int(ep) not in excluded_set]

    longest_dropped = []
    if drop_longest_n > 0:
        if len(remaining_eps) <= drop_longest_n:
            raise ValueError(
                f"可用 episode 數 ({len(remaining_eps)}) 不足，無法排除最長的 {drop_longest_n} 筆。"
            )
        ep_durations = []
        for ep in remaining_eps:
            m = ep_masks[ep]
            n_frames = int(np.sum(m))
            if timestamps is not None and n_frames > 1:
                t_start = float(np.asarray(timestamps[m][0]).item())
                t_end = float(np.asarray(timestamps[m][-1]).item())
                dur = t_end - t_start
            else:
                dur = float(n_frames)
            ep_durations.append((dur, n_frames, ep))
        ep_durations.sort(key=lambda x: (x[0], x[1], x[2]), reverse=True)
        for dur, n_frames, ep in ep_durations[:drop_longest_n]:
            excluded_set.add(ep)
            longest_dropped.append((ep, n_frames, dur))

    valid_indices = []
    dropped_ep_frames = 0
    trimmed_head_frames = 0
    trimmed_tail_frames = 0

    for ep in unique_eps:
        ep_int = int(ep)
        m = ep_masks[ep_int]
        ep_idx = indices[m]
        if ep_int in excluded_set:
            dropped_ep_frames += len(ep_idx)
            continue

        ep_states = states[m]
        n_ep = len(ep_states)

        if trim_head:
            head_diffs = np.max(np.abs(ep_states - ep_states[0]), axis=1)
            head_moved = np.where(head_diffs > move_thresh)[0]
            start_offset = max(0, int(head_moved[0]) - keep_lead_frames) if len(head_moved) > 0 else 0
        else:
            start_offset = 0

        if trim_tail:
            tail_diffs = np.max(np.abs(ep_states - ep_states[-1]), axis=1)
            tail_moved = np.where(tail_diffs > move_thresh)[0]
            end_offset = (
                min(n_ep, int(tail_moved[-1]) + 1 + keep_tail_frames)
                if len(tail_moved) > 0
                else n_ep
            )
        else:
            end_offset = n_ep

        if end_offset <= start_offset:
            start_offset = 0
            end_offset = n_ep

        trimmed_head_frames += start_offset
        trimmed_tail_frames += n_ep - end_offset
        valid_indices.extend(ep_idx[start_offset:end_offset].tolist())

    if longest_dropped:
        dropped_desc = ", ".join(
            f"ep{ep}({n}f/{dur:.1f}s)" if timestamps is not None else f"ep{ep}({n}f)"
            for ep, n, dur in longest_dropped
        )
        print(f"Excluded longest {len(longest_dropped)} episodes: {dropped_desc}")
    if len(excluded_set) > len(longest_dropped):
        manual_eps = sorted(excluded_set - {ep for ep, _, _ in longest_dropped})
        print(f"Excluded manual episodes: {manual_eps}")

    removed = len(df) - len(valid_indices)
    print(
        f"Dataset filter summary: {len(df)} -> {len(valid_indices)} frames "
        f"(kept {len(valid_indices) / len(df) * 100:.1f}% | "
        f"dropped_eps={dropped_ep_frames}, trimmed_head={trimmed_head_frames}, "
        f"trimmed_tail={trimmed_tail_frames}, total_removed={removed})"
    )
    return torch.utils.data.Subset(dataset, valid_indices)

exit_reason = None
last_completed_step = start_step

if start_step >= STEPS:
    print(f"Training already completed ({start_step}/{STEPS} steps). Skipping loop.")
    exit_reason = "completed"
else:
    train_ds = build_trimmed_subset(
        ds,
        DS_ROOT,
        move_thresh=3.0,
        keep_lead_frames=10,
        keep_tail_frames=10,
        trim_head=True,
        trim_tail=True,
        drop_longest_n=0,
        exclude_episodes=(),
    )
    loader = torch.utils.data.DataLoader(
        train_ds,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True,
        num_workers=NUM_WORKERS,
        pin_memory=(device == "cuda" and NUM_WORKERS > 0),
    )
    policy.train()
    it = iter(loader)
    session_start_t = time.time()

    try:
        for step in range(start_step + 1, STEPS + 1):
            try:
                batch = next(it)
            except StopIteration:
                it = iter(loader)
                batch = next(it)

            batch = preprocess(batch)
            out = policy(batch)
            loss = out[0] if isinstance(out, (tuple, list)) else out["loss"]

            optimizer.zero_grad()
            loss.backward()
            if grad_clip_norm and grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(policy.parameters(), grad_clip_norm)
            optimizer.step()
            scheduler.step()

            loss_val = loss.detach().item()
            losses.append(loss_val)
            last_completed_step = step
            elapsed_min = (time.time() - session_start_t) / 60.0

            if step % 200 == 0 or step == start_step + 1:
                cur_lr = optimizer.param_groups[0]["lr"]
                print(f"step {step:5d}/{STEPS} | loss {loss_val:.4f} | lr {cur_lr:.2e} | elapsed {elapsed_min:.1f}m")

            reach_limit = (
                MAX_SESSION_MINUTES is not None
                and elapsed_min >= MAX_SESSION_MINUTES
                and step < STEPS
            )
            save_targets = []
            if step % SAVE_LATEST_FREQ == 0 or step == STEPS or reach_limit:
                save_targets.append(LATEST_DIR)
            if step % MILESTONE_FREQ == 0 or step == STEPS:
                save_targets.append(os.path.join(DRIVE_CHECKPOINT_ROOT, f"smolvla_pick_step_{step}"))

            if save_targets:
                save_checkpoint(save_targets, step)
                for t_dir in save_targets:
                    print(f"--> Checkpoint synced & verified at step {step}: {t_dir}")

            if reach_limit:
                exit_reason = "session_limit"
                print(
                    f"\n[Session Limit] Reached {elapsed_min:.1f}m (>= {MAX_SESSION_MINUTES}m). "
                    f"Saved progress at step {step}/{STEPS} to {LATEST_DIR}."
                )
                break
        else:
            exit_reason = "completed"
    except KeyboardInterrupt:
        exit_reason = "interrupted"
        if last_completed_step > start_step:
            save_checkpoint([LATEST_DIR], last_completed_step)
            print(f"\n[Interrupted] Saved & verified progress at step {last_completed_step}/{STEPS} to {LATEST_DIR}.")
    except Exception as e:
        exit_reason = "error"
        if last_completed_step > start_step:
            try:
                save_checkpoint([LATEST_DIR], last_completed_step)
                print(f"\n[Error Recovery] Saved progress at step {last_completed_step}/{STEPS} to {LATEST_DIR}.")
            except Exception as save_err:
                print(f"\n[Error Recovery] Emergency save failed: {save_err}")
        raise

if losses:
    plt.figure(figsize=(8, 3))
    plt.plot(range(1, len(losses) + 1), losses, alpha=0.7)
    plt.xlabel("step")
    plt.ylabel("flow-matching loss")
    plt.title("SmolVLA Fine-tuning Loss (so101_pick_cube)")
    plt.yscale("log")
    plt.grid(True)
    plt.show()

if exit_reason in ("completed", "session_limit"):
    policy.eval()
    policy.reset()
    sample = ds[0]
    obs_batch = {
        k: v.unsqueeze(0)
        for k, v in sample.items()
        if k.startswith("observation.") and isinstance(v, torch.Tensor)
    }
    obs_batch["task"] = sample.get("task", "Pick up the stylus from the stand")
    with torch.no_grad():
        pred_action = postprocess(policy.select_action(preprocess(obs_batch)))
    print("Sanity check - predicted action shape:", tuple(pred_action.shape))
    print("Sanity check - first predicted step:", pred_action.squeeze(0).detach().cpu().numpy().round(3))

    final_milestone = os.path.join(DRIVE_CHECKPOINT_ROOT, f"smolvla_pick_step_{STEPS}")
    verify_target = (
        final_milestone if (exit_reason == "completed" and verify_checkpoint_dir(final_milestone))
        else LATEST_DIR
    )
    if not verify_checkpoint_dir(verify_target):
        raise RuntimeError(f"雲端最終檢查失敗，找不到完整模型檔案：{verify_target}")

    weights_mb = os.path.getsize(os.path.join(verify_target, "model.safetensors")) / (1024 * 1024)
    print(f"[Cloud Verified] 模型已完整儲存至 Google Drive：{verify_target} (model.safetensors: {weights_mb:.1f} MB)")

    should_disconnect = (
        (exit_reason == "completed" and AUTO_DISCONNECT_ON_COMPLETE)
        or (exit_reason == "session_limit" and AUTO_DISCONNECT_ON_SESSION_LIMIT)
    )
    if should_disconnect:
        os.sync()
        try:
            from google.colab import drive, runtime
            print("[Cloud Sync] Flushing Google Drive writes before disconnecting...")
            drive.flush_and_unmount()
            print("[Runtime] Disconnecting Colab runtime now...")
            runtime.unassign()
        except ImportError:
            print("[Runtime] 非 Colab 環境，略過 flush_and_unmount / runtime.unassign。")